In [2]:
import numpy as np
from collections import Counter
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)

# Read from LAMMPS data file
ld = LammpsData.from_file('bulk_structure/mg_unitcell_relaxed.data', atom_style='atomic')
alloy = AseAtomsAdaptor().get_atoms(ld.structure)
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")
print(f"Cell: {np.round(alloy.cell.lengths(), 4)} A")
print(f"Angles: {np.round(alloy.cell.angles(), 2)} deg")

Bulk: 2 atoms, Mg2
Cell: [3.2091 3.2091 5.1969] A
Angles: [ 90.  90. 120.] deg


In [5]:
# (0001) basal plane = (0,0,1) in 3-index
# Start with 6 layers — for hcp Mg the (0001) d-spacing is c/2 ≈ 2.6 A
# so 6 layers ≈ 15.6 A thick

slab = surface(alloy, (0, 0, 1), 8, vacuum=8)

# For pure elements, 1x1 might be small — check cell dimensions
print(f"Atoms: {len(slab)}")
print(f"Cell: {np.round(slab.cell.lengths(), 3)} A")

# Surface area
area = np.linalg.norm(np.cross(slab.cell[0], slab.cell[1]))
print(f"Surface area: {area:.2f} A^2")

# If area is small, supercell
if area < 50:
    print("Small area — making 3x3 supercell")
    slab = make_supercell(slab, [[3,0,0],[0,3,0],[0,0,1]])
    area = np.linalg.norm(np.cross(slab.cell[0], slab.cell[1]))
    print(f"New: {len(slab)} atoms, area={area:.2f} A^2")

z = slab.get_positions()[:, 2]
thick = z.max() - z.min()
print(f"Thickness: {thick:.2f} A")

# Check symmetry
ps = AseAtomsAdaptor().get_structure(slab)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(0,0,1), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 16
Cell: [ 3.209  3.209 54.977] A
Surface area: 8.92 A^2
Small area — making 3x3 supercell
New: 144 atoms, area=80.27 A^2
Thickness: 38.98 A
Symmetric: True


In [6]:
view(slab)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [7]:
ps = AseAtomsAdaptor().get_structure(slab)
ld_out = LammpsData.from_structure(ps, atom_style='atomic')
ld_out.write_file("slabs/Mg_0001_8L.data")

z = slab.get_positions()[:, 2]
thick = z.max() - z.min()
area = np.linalg.norm(np.cross(slab.cell[0], slab.cell[1]))

print(f"Saved: slabs/Mg_0001_8L.data")
print(f"  Atoms: {len(slab)}")
print(f"  Thickness: {thick:.2f} A")
print(f"  Area: {area:.2f} A^2")
print(f"  Symmetric: True")

Saved: slabs/Mg_0001_8L.data
  Atoms: 144
  Thickness: 38.98 A
  Area: 80.27 A^2
  Symmetric: True
